# Transform

Aplica todas las transformaciones identificadas en el EDA: features temporales, ingeniería de features, imputación de nulos, encoding de categóricas y normalización de eventos.

Input: `data_raw.parquet`  
Output: `data_transformed.parquet`

---

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/raw/data_raw.parquet')
print('Shape:', df.shape)
df.head()

## 1. Features temporales

El EDA mostró que la hora de instalación y el día de la semana tienen señal: churn alto en madrugada y especialmente el sábado.

In [2]:
df['install_time'] = pd.to_datetime(df['install_time'])

df['install_hour'] = df['install_time'].dt.hour
df['install_dow']  = df['install_time'].dt.dayofweek  # 0=lunes, 6=domingo
df['is_weekend']   = df['install_dow'].isin([5, 6]).astype(int)

print('Features temporales creadas: install_hour, install_dow, is_weekend')
df[['install_hour', 'install_dow', 'is_weekend']].describe()

Features temporales creadas: install_hour, install_dow, is_weekend


,install_hour,install_dow,is_weekend
count,18533.000000,18533.000000,18533.000000
mean,12.102736,3.306265,0.394324
std,8.268568,2.011832,0.488718
min,0.000000,0.000000,0.000000
25%,3.000000,1.000000,0.000000
50%,15.000000,4.000000,0.000000
75%,19.000000,5.000000,1.000000
max,23.000000,6.000000,1.000000


## 2. Features de eventos

Dos transformaciones identificadas en el EDA:
- `event_3` tiene 74.7% de ceros — se crea una versión binaria `event_3_used`.
- Todas las distribuciones son muy sesgadas a la derecha — se aplica `log1p` para reducir el efecto de outliers.

In [3]:
events = ['event_1', 'event_2', 'event_3', 'event_4', 'event_5']

df['event_3_used'] = (df['event_3'] > 0).astype(int)

for col in events:
    df[f'{col}_log'] = np.log1p(df[col])

print('event_3_used:', df['event_3_used'].value_counts().to_dict())
df[[f'{e}_log' for e in events]].describe().round(2)

event_3_used: {0: 13851, 1: 4682}


,event_1_log,event_2_log,event_3_log,event_4_log,event_5_log
count,18533.00,18533.00,18533.00,18533.00,18533.00
mean,2.06,2.30,0.26,1.44,1.46
std,1.14,1.24,0.49,0.66,0.71
min,0.00,0.00,0.00,0.00,0.00
25%,1.10,1.39,0.00,0.69,1.10
50%,1.95,2.30,0.00,1.39,1.39
75%,2.94,3.22,0.69,1.95,1.95
max,5.39,6.08,3.04,3.78,4.17


## 3. Limpieza de edad

El dataset tiene 15 registros con rangos de edad inválidos (min > max) y 9 con rangos puntuales (ej: 13-13). Se eliminan por ser ruido marginal (<30 registros en total).

Con los dos grupos válidos (13-17 y 18-20) se crea una feature binaria `is_teen`.

In [4]:
invalidos = df['min_age_range'] >= df['max_age_range']
print(f'Registros con rango de edad inválido o puntual: {invalidos.sum()}')
df = df[~invalidos].reset_index(drop=True)

df['is_teen'] = (df['min_age_range'] == 13).astype(int)  # 13-17 vs 18-20

print('Shape después de filtrar:', df.shape)
print('is_teen:', df['is_teen'].value_counts().to_dict())

Registros con rango de edad inválido o puntual: 24
Shape después de filtrar: (18509, 24)
is_teen: {0: 11593, 1: 6916}


## 4. Imputación de nulos

Los nulos en variables geográficas se imputan con `"unknown"`, que el encoder posterior va a tratar como una categoría más.

In [5]:
df['country_region'] = df['country_region'].fillna('unknown')
df['city']           = df['city'].fillna('unknown')

print('Nulos restantes:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Nulos restantes:
Series([], dtype: int64)


## 5. Encoding de variables categóricas

- `platform`: binario (iOS=1, Android=0)
- `gender`: label encoding (female=0, male=1, unknown=2)
- `country_region`: label encoding — ~20 categorías, manejable
- `city`: se descarta por alta cardinalidad (cientos de ciudades únicas, poco aporte marginal sobre región)

In [6]:
df['platform_enc'] = (df['platform'] == 'iOS').astype(int)

gender_map = {'female': 0, 'male': 1, 'unknown': 2}
df['gender_enc'] = df['gender'].map(gender_map)

df['region_enc'], region_index = pd.factorize(df['country_region'])
print('Regiones codificadas:', len(region_index))
print(dict(enumerate(region_index)))

Regiones codificadas: 32
{0: 'Santa Cruz', 1: 'Corrientes', 2: 'Cordoba', 3: 'Buenos Aires', 4: 'Santiago del Estero', 5: 'Entre Rios', 6: 'Mendoza', 7: 'Buenos Aires F.D.', 8: 'Santa Fe', 9: 'Salta', 10: 'Misiones', 11: 'Neuquen', 12: 'La Pampa', 13: 'Jujuy', 14: 'Chaco', 15: 'unknown', 16: 'Tucuman', 17: 'Chubut', 18: 'Formosa', 19: 'San Juan', 20: 'Tierra del Fuego', 21: 'La Rioja', 22: 'Entre Ríos Province', 23: 'San Luis', 24: 'Rio Negro', 25: 'Catamarca', 26: 'Neuquén Province', 27: 'Cordoba Province', 28: 'Salta Province', 29: 'Tucumán Province', 30: 'Chubut Province', 31: 'Río Negro Province'}


## 6. Selección de features finales

Se descartan columnas originales que ya fueron transformadas o que no aportan al modelo.

In [7]:
cols_drop = [
    'install_time',
    'platform',
    'gender',
    'country_region',
    'city',
    'min_age_range',
    'max_age_range',
    *events  # reemplazados por versiones log
]

df = df.drop(columns=cols_drop)

print('Shape final:', df.shape)
print('Columnas:', df.columns.tolist())

Shape final: (18509, 15)
Columnas: ['user_id', 'target_churn_indicator', 'install_hour', 'install_dow', 'is_weekend', 'event_3_used', 'event_1_log', 'event_2_log', 'event_3_log', 'event_4_log', 'event_5_log', 'is_teen', 'platform_enc', 'gender_enc', 'region_enc']


## 7. Verificación final

In [8]:
print('Nulos:', df.isnull().sum().sum())
print('\nTipos de datos:')
print(df.dtypes)
df.head()

Nulos: 0

Tipos de datos:
user_id                    object
target_churn_indicator      int64
install_hour                int32
install_dow                 int32
is_weekend                  int64
event_3_used                int64
event_1_log               float64
event_2_log               float64
event_3_log               float64
event_4_log               float64
event_5_log               float64
is_teen                     int64
platform_enc                int64
gender_enc                  int64
region_enc                  int64
dtype: object


,user_id,target_churn_indicator,install_hour,install_dow,is_weekend,event_3_used,event_1_log,event_2_log,event_3_log,event_4_log,event_5_log,is_teen,platform_enc,gender_enc,region_enc
0,2ba6f357,0,19,2,0,0,0.693147,1.098612,0.0,0.693147,0.693147,1,0,1,0
1,3cb936c1,1,13,5,1,0,0.000000,2.079442,0.0,1.386294,0.000000,1,0,1,1
2,17d88bbc,1,16,5,1,0,1.098612,0.693147,0.0,1.098612,1.098612,0,0,0,2
3,7baa10b6,0,21,0,0,0,1.791759,1.945910,0.0,1.098612,1.791759,1,0,0,3
4,120d379,0,4,5,1,0,2.197225,2.564949,0.0,1.098612,1.386294,1,0,0,4


## 8. Export

In [ ]:
df.to_parquet('../data/processed/data_transformed.parquet', index=False)
print('Guardado en data/processed/data_transformed.parquet')